# Assignment 2: Project Text Classification Models
**Course**: Big Data Analytics (BDA) — 6th Semester  
**Domain**: HDA-4 | **Group**: 4  

**Team Members**:  
- Krishna Sikheriya (IIT2023139)  
- Priyam Jyoti Chakrabarty (IIT2023147)  
- Tavish Chawla (IIT2023150)  

**Objective**: Develop classification models to detect MDD posts using Classical ML (TF-IDF) and Deep Learning NLP (Bio_ClinicalBERT) on the `reddit_mdd_cleaned.csv` dataset.

## 0. Google Colab & GitHub Sync
This cell dynamically connects Colab to your GitHub repository utilizing Colab Secrets. It pulls the latest commits, allowing the notebook to fetch live CSV data securely from `../data/`.

In [ ]:
import os
import sys

# Detects if running inside Google Colab environment
if 'google.colab' in sys.modules:
    from google.colab import userdata
    # Grabs the Personal Access Token securely stored in Colab Secrets panel
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Krishna200608/BDA-MDD-Reddit-NLP.git"
    
    # Setup explicit author identity for push/pull loops
    !git config --global user.email "krishnasikheriya001@gmail.com"
    !git config --global user.name "Krishna200608"
    
    # Clone only if not present
    if not os.path.exists('BDA-MDD-Reddit-NLP'):
        !git clone {REPO_URL}
        
    # Move down to notebook directory to perfectly emulate local relative routing (e.g. `../data/processed`)
    os.chdir('BDA-MDD-Reddit-NLP/notebooks')
    print("✅ Successfully synced Colab directly into live GitHub deployment tree.")
else:
    print("💻 Running seamlessly on Local PC terminal path.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Ensure plots look professional
sns.set_theme(style="whitegrid")
tqdm.pandas()

## 1. Load the Processed Dataset
We load the clean NLP dataset generated purely via `pipeline.py` / `scraper.py`.

In [ ]:
df = pd.read_csv('../data/processed/reddit_mdd_cleaned.csv')
print(f"Dataset Size: {df.shape}")

# Drop any residual missing NaNs that might have squeezed through CSV saving
df = df.dropna(subset=['selftext_cleaned']).reset_index(drop=True)

# Binary map our target labels: MDD = 1, Control = 0
df['target'] = df['label'].map({'MDD': 1, 'Control': 0})
df['target'].value_counts()

## 2. Baseline Model: TF-IDF + Logistic Regression
We establish our lower-bound baseline metrics utilizing term-frequencies. This operates instantly on the CPU.

In [ ]:
# Split the data globally for all models
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['selftext_cleaned'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Vectorize Text
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_tfidf, y_train)

# Predict & Metrics
y_pred_lr = lr_model.predict(X_test_tfidf)
print("Logistic Regression (TF-IDF) Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

# Visual Matrix
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=['Control', 'MDD'], yticklabels=['Control', 'MDD'])
plt.title('Baseline: Logistic Regression Confusion Matrix')
plt.show()

## 3. Advanced NLP Method: Embedding Generation (Bio_ClinicalBERT)
To capture deep semantic context surrounding medical keywords and depressive syntax, we encode the texts using HuggingFace's `emilyalsentzer/Bio_ClinicalBERT`.

> **Hardware Note:** Standard CPUs parse Transformers relatively slowly compared to NVIDIA CUDA. This specific cell dynamically routes to CUDA acceleration if you are using Google Colab's T4 GPU, rendering the entire dataset instantly. If you are running locally without an NVIDIA GPU, this cell detects it and gracefully scales down to fetch exactly 2000 items to fit perfectly within standard PC runtime margins.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# Setup Model Ecosystem with dynamic hardware routing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Accelerating ClinicalBERT Embeddings via: {device}")

model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)
bert_model.eval() # Set purely to inference mode to save RAM

# If using Colab GPU, we can comfortably process all 10k rows.
# If running locally on CPU, we sample 2000 to save time.
n_samples = len(df) if torch.cuda.is_available() else 2000
df_bert_sample = df.sample(n=n_samples, random_state=42).reset_index(drop=True)
print(f"Subset processing for BERT generation: {df_bert_sample.shape[0]} documents")

In [ ]:
def get_bert_embedding(text):
    # Standardize input lengths to 512 max BERT tokens and pipe to correct hardware
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    # Extract [CLS] hidden state, yank it back to CPU format for Numpy ML parsing
    return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print("Generating Bio_ClinicalBERT Embeddings... (This dynamically routes via PyTorch!)")
bert_embeddings = df_bert_sample['selftext_cleaned'].progress_apply(get_bert_embedding)
# Stack the Series of arrays into a rigid 2D numpy matrix mapping
X_bert = np.vstack(bert_embeddings.values)
y_bert = df_bert_sample['target'].values

## 4. Run Machine Learning Pipeline on ClinicalBERT Vectors
Now we pass the massive 768-dimension continuous feature vectors structurally to a Random Forest classifier.

In [ ]:
# Spilt the BERT Sample
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bert, y_bert, test_size=0.2, random_state=42, stratify=y_bert
)

# We use Random Forest because continuous multi-dimensional embeddings work perfectly with tree depth models.
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_b, y_train_b)

y_pred_rf = rf_model.predict(X_test_b)
print("ClinicalBERT + Random Forest Accuracy:", accuracy_score(y_test_b, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test_b, y_pred_rf))

cm_rf = confusion_matrix(y_test_b, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'MDD'], yticklabels=['Control', 'MDD'])
plt.title('ClinicalBERT: Random Forest Confusion Matrix')
plt.show()